# Supervised Fusion Baseline: Point 3

This notebook trains and evaluates the supervised multimodal fusion baseline on the cached windows from notebook `01_segment_index_and_windows.ipynb`.

It assumes the full point-2 cache has already been materialized under `artifacts/windows/`. If it has not, open notebook 01, set `BUILD_FULL_WINDOW_CACHE = True`, and run the cache-build cell first.

The baseline follows the plan:
- input: normalized fused windows of shape `(12, 60)`
- model: 1D CNN encoder + linear classifier
- validation: subject-held-out validation from Kaggle train subjects
- outputs: checkpoint, metrics JSON, confusion matrix, and per-subject accuracy table

In [ ]:
from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
import csv
import json
import math
import pickle
import random
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

if (Path.cwd() / "artifacts").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "artifacts").exists():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError("Could not locate the project root containing artifacts/")

ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
WINDOW_ROOT = ARTIFACT_ROOT / "windows"
SUPERVISED_ROOT = ARTIFACT_ROOT / "supervised"
SUPERVISED_ROOT.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = WINDOW_ROOT / "manifest.json"
HAS_WINDOW_CACHE = MANIFEST_PATH.exists()
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if HAS_WINDOW_CACHE else None
LABEL_TO_INDEX = manifest["label_to_index"] if manifest is not None else None
INDEX_TO_LABEL = {index: label for label, index in LABEL_TO_INDEX.items()} if LABEL_TO_INDEX else None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Torch device: {DEVICE}")
if manifest is not None:
    print(f"Window counts: {manifest['split_window_counts']}")
else:
    print("Window cache manifest not found yet. Smoke mode can bootstrap a small in-memory dataset directly from notebook 01.")

In [ ]:
SEED = 42
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
FULL_TRAIN_EPOCHS = 40
SMOKE_TRAIN_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 7
DROPOUT = 0.2

# By default the notebook runs a small smoke path to validate the pipeline.
RUN_SMOKE_TRAIN = True
RUN_FULL_TRAIN = True
ALLOW_DIRECT_SMOKE_BOOTSTRAP = True
SMOKE_LIMITS = {
    "train": 2048,
    "val": 512,
    "test": 1024,
}

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
torch.set_num_threads(max(1, min(8, torch.get_num_threads())))

if RUN_FULL_TRAIN and not HAS_WINDOW_CACHE:
    raise FileNotFoundError(
        "Full training requires the point-2 cache. Run notebook 01 with BUILD_FULL_WINDOW_CACHE = True first."
    )

In [ ]:
def read_metadata_rows(split: str) -> List[dict]:
    metadata_path = WINDOW_ROOT / split / "metadata.csv"
    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing metadata for split {split}: {metadata_path}")
    with metadata_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        return [row for row in reader]


def read_window_payloads(split: str) -> Dict[int, List[List[float]]]:
    split_dir = WINDOW_ROOT / split
    chunk_paths = sorted(split_dir.glob("data_chunk_*.pkl"))
    if not chunk_paths:
        raise FileNotFoundError(f"No data chunks found for split {split}: {split_dir}")

    payloads: Dict[int, List[List[float]]] = {}
    for chunk_path in chunk_paths:
        with chunk_path.open("rb") as handle:
            chunk_records = pickle.load(handle)
        for record in chunk_records:
            payloads[int(record["window_id"])] = record["x_fusion"]
    return payloads


def load_notebook01_namespace() -> dict:
    notebook_path = PROJECT_ROOT / "notebooks" / "01_segment_index_and_windows.ipynb"
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    namespace: dict = {}
    for index, cell in enumerate(notebook["cells"]):
        if cell["cell_type"] != "code":
            continue
        if index == 6:
            break
        exec(compile("".join(cell["source"]), f"{notebook_path}#cell{index}", "exec"), namespace)
    return namespace


def load_direct_smoke_arrays(limit_by_split: Dict[str, int]) -> Tuple[dict, Dict[str, Tuple[np.ndarray, np.ndarray, np.ndarray, List[dict]]]]:
    namespace = load_notebook01_namespace()
    subject_splits = namespace["build_dataset_subject_splits"](namespace["DATA_ROOT"])
    label_to_index = namespace["compute_label_mapping"](subject_splits)
    sampled_by_split = {}

    def reservoir_sample_windows(split: str, subject_id: int, quota: int, seed: int):
        rng = random.Random(seed)
        sample = []
        seen = 0
        for item in namespace["generate_windows_for_subject"](namespace["DATA_ROOT"], split, subject_id):
            seen += 1
            if len(sample) < quota:
                sample.append(item)
            else:
                replacement_index = rng.randrange(seen)
                if replacement_index < quota:
                    sample[replacement_index] = item
        return sample

    for split in ["train", "val", "test"]:
        split_samples = []
        subject_ids = list(subject_splits[split])
        per_subject_quota = max(1, math.ceil(limit_by_split[split] / max(1, len(subject_ids))))
        for subject_id in subject_ids:
            subject_samples = reservoir_sample_windows(split, subject_id, per_subject_quota, seed=SEED + subject_id)
            split_samples.extend(subject_samples)
        split_samples = split_samples[: limit_by_split[split]]
        sampled_by_split[split] = split_samples

    means = namespace["accumulate_channel_stats"](sampled_by_split["train"])

    arrays_by_split = {}
    window_id = 0
    for split in ["train", "val", "test"]:
        x_list = []
        y_list = []
        subject_list = []
        metadata_rows = []
        for metadata, fused_channels in sampled_by_split[split]:
            normalized_window = namespace["normalize_window"](fused_channels, means["means"], means["stds"])
            metadata = dict(metadata)
            metadata["window_id"] = window_id
            metadata["label_idx"] = label_to_index[metadata["activity_label"]]
            x_list.append(normalized_window)
            y_list.append(metadata["label_idx"])
            subject_list.append(metadata["subject_id"])
            metadata_rows.append(metadata)
            window_id += 1

        arrays_by_split[split] = (
            np.asarray(x_list, dtype=np.float32),
            np.asarray(y_list, dtype=np.int64),
            np.asarray(subject_list, dtype=np.int64),
            metadata_rows,
        )

    bootstrap_manifest = {
        "target_hz": namespace["TARGET_HZ"],
        "window_seconds": namespace["WINDOW_SECONDS"],
        "window_stride_seconds": namespace["WINDOW_STRIDE_SECONDS"],
        "window_length": namespace["WINDOW_LENGTH"],
        "label_to_index": label_to_index,
        "subject_splits": subject_splits,
        "normalization": {
            "means": means["means"],
            "stds": means["stds"],
            "train_window_count": means["window_count"],
        },
        "split_window_counts": {split: int(arrays_by_split[split][0].shape[0]) for split in arrays_by_split},
        "source": "direct_smoke_bootstrap",
    }
    return bootstrap_manifest, arrays_by_split


def load_split_arrays(split: str, limit: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[dict]]:
    metadata_rows = read_metadata_rows(split)
    payloads = read_window_payloads(split)
    if limit is not None and len(metadata_rows) > limit:
        rng = random.Random(SEED + sum(ord(char) for char in split))
        selected_indices = sorted(rng.sample(range(len(metadata_rows)), limit))
        metadata_rows = [metadata_rows[index] for index in selected_indices]

    x_list = []
    y_list = []
    subject_list = []
    for row in metadata_rows:
        window_id = int(row["window_id"])
        x_list.append(payloads[window_id])
        y_list.append(int(row["label_idx"]))
        subject_list.append(int(row["subject_id"]))

    x = np.asarray(x_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.int64)
    subjects = np.asarray(subject_list, dtype=np.int64)
    return x, y, subjects, metadata_rows


def maybe_limit_split(split: str) -> Optional[int]:
    if RUN_FULL_TRAIN:
        return None
    if RUN_SMOKE_TRAIN:
        return SMOKE_LIMITS[split]
    return None


bootstrap_arrays = None
if HAS_WINDOW_CACHE:
    x_train, y_train, subjects_train, train_rows = load_split_arrays("train", maybe_limit_split("train"))
    x_val, y_val, subjects_val, val_rows = load_split_arrays("val", maybe_limit_split("val"))
    x_test, y_test, subjects_test, test_rows = load_split_arrays("test", maybe_limit_split("test"))
elif RUN_SMOKE_TRAIN and ALLOW_DIRECT_SMOKE_BOOTSTRAP:
    manifest, bootstrap_arrays = load_direct_smoke_arrays(SMOKE_LIMITS)
    LABEL_TO_INDEX = manifest["label_to_index"]
    INDEX_TO_LABEL = {index: label for label, index in LABEL_TO_INDEX.items()}
    x_train, y_train, subjects_train, train_rows = bootstrap_arrays["train"]
    x_val, y_val, subjects_val, val_rows = bootstrap_arrays["val"]
    x_test, y_test, subjects_test, test_rows = bootstrap_arrays["test"]
else:
    raise FileNotFoundError(
        "No window cache found. Run notebook 01 with BUILD_FULL_WINDOW_CACHE = True, or keep smoke bootstrap enabled."
    )

print("Loaded arrays:")
for split_name, x, y in [("train", x_train, y_train), ("val", x_val, y_val), ("test", x_test, y_test)]:
    print(split_name, x.shape, y.shape)

assert x_train.ndim == 3 and x_train.shape[1:] == (12, 60)
assert x_val.ndim == 3 and x_val.shape[1:] == (12, 60)
assert x_test.ndim == 3 and x_test.shape[1:] == (12, 60)
assert not np.isnan(x_train).any()
assert y_train.min() >= 0 and y_train.max() < len(LABEL_TO_INDEX)

In [ ]:
def to_loader(x: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(x), torch.from_numpy(y))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


train_loader = to_loader(x_train, y_train, BATCH_SIZE, shuffle=True)
val_loader = to_loader(x_val, y_val, BATCH_SIZE, shuffle=False)
test_loader = to_loader(x_test, y_test, BATCH_SIZE, shuffle=False)


class TimeSeriesEncoder(nn.Module):
    def __init__(self, in_channels: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        return x.squeeze(-1)


class SupervisedHARModel(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.encoder = TimeSeriesEncoder(in_channels=12)
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.encoder(x)
        return self.classifier(features)


model = SupervisedHARModel(num_classes=len(LABEL_TO_INDEX)).to(DEVICE)
sample_logits = model(torch.from_numpy(x_train[:4]).to(DEVICE))
print("Sample logits shape:", tuple(sample_logits.shape))
assert tuple(sample_logits.shape) == (4, len(LABEL_TO_INDEX))

In [ ]:
def confusion_matrix_from_predictions(y_true: Sequence[int], y_pred: Sequence[int], num_classes: int) -> np.ndarray:
    matrix = np.zeros((num_classes, num_classes), dtype=np.int64)
    for truth, pred in zip(y_true, y_pred):
        matrix[int(truth), int(pred)] += 1
    return matrix


def macro_f1_from_confusion(matrix: np.ndarray) -> Tuple[float, Dict[str, float]]:
    f1_by_label: Dict[str, float] = {}
    scores = []
    for class_index in range(matrix.shape[0]):
        tp = float(matrix[class_index, class_index])
        fp = float(matrix[:, class_index].sum() - tp)
        fn = float(matrix[class_index, :].sum() - tp)
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2.0 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        f1_by_label[INDEX_TO_LABEL[class_index]] = f1
        scores.append(f1)
    return float(sum(scores) / len(scores)), f1_by_label


def evaluate_model(
    model: nn.Module,
    loader: DataLoader,
    loss_fn: nn.Module,
    subjects: np.ndarray,
) -> dict:
    model.eval()
    total_loss = 0.0
    all_true: List[int] = []
    all_pred: List[int] = []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            total_loss += float(loss.item()) * len(x_batch)
            predictions = logits.argmax(dim=1)
            all_true.extend(y_batch.cpu().tolist())
            all_pred.extend(predictions.cpu().tolist())

    matrix = confusion_matrix_from_predictions(all_true, all_pred, num_classes=len(LABEL_TO_INDEX))
    macro_f1, per_class_f1 = macro_f1_from_confusion(matrix)
    accuracy = float((matrix.diagonal().sum()) / max(1, matrix.sum()))

    per_subject_counts: Dict[int, List[int]] = defaultdict(lambda: [0, 0])
    for subject_id, truth, pred in zip(subjects.tolist(), all_true, all_pred):
        per_subject_counts[subject_id][0] += int(truth == pred)
        per_subject_counts[subject_id][1] += 1
    per_subject_accuracy = {
        int(subject_id): correct / total
        for subject_id, (correct, total) in sorted(per_subject_counts.items())
    }

    return {
        "loss": total_loss / len(all_true),
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "per_class_f1": per_class_f1,
        "confusion_matrix": matrix,
        "per_subject_accuracy": per_subject_accuracy,
        "y_true": all_true,
        "y_pred": all_pred,
    }


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, loss_fn: nn.Module) -> float:
    model.train()
    total_loss = 0.0
    total_count = 0
    for x_batch, y_batch in loader:
        x_batch = x_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += float(loss.item()) * len(x_batch)
        total_count += len(x_batch)
    return total_loss / max(1, total_count)

In [ ]:
def plot_confusion_matrix(matrix: np.ndarray, title: str, output_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(10, 8))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ticks = np.arange(len(INDEX_TO_LABEL))
    labels = [INDEX_TO_LABEL[index] for index in ticks]
    ax.set_xticks(ticks)
    ax.set_xticklabels(labels)
    ax.set_yticks(ticks)
    ax.set_yticklabels(labels)
    fig.colorbar(image, ax=ax)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150)
    plt.close(fig)


def train_supervised_baseline() -> dict:
    seed_everything(SEED)
    model = SupervisedHARModel(num_classes=len(LABEL_TO_INDEX)).to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    max_epochs = FULL_TRAIN_EPOCHS if RUN_FULL_TRAIN else SMOKE_TRAIN_EPOCHS
    best_val_macro_f1 = -1.0
    best_state = None
    best_epoch = -1
    epochs_without_improvement = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
        val_metrics = evaluate_model(model, val_loader, loss_fn, subjects_val)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_metrics["loss"],
                "val_accuracy": val_metrics["accuracy"],
                "val_macro_f1": val_metrics["macro_f1"],
            }
        )
        print(
            f"epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_metrics['loss']:.4f} "
            f"val_acc={val_metrics['accuracy']:.4f} val_macro_f1={val_metrics['macro_f1']:.4f}"
        )

        if val_metrics["macro_f1"] > best_val_macro_f1:
            best_val_macro_f1 = val_metrics["macro_f1"]
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if RUN_FULL_TRAIN and epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch}")
                break

    if best_state is None:
        raise RuntimeError("Training completed without a best checkpoint")

    model.load_state_dict(best_state)
    train_metrics = evaluate_model(model, train_loader, loss_fn, subjects_train)
    val_metrics = evaluate_model(model, val_loader, loss_fn, subjects_val)
    test_metrics = evaluate_model(model, test_loader, loss_fn, subjects_test)

    run_name = "full" if RUN_FULL_TRAIN else "smoke"
    checkpoint_path = SUPERVISED_ROOT / f"supervised_baseline_{run_name}.pt"
    metrics_path = SUPERVISED_ROOT / f"metrics_{run_name}.json"
    confusion_path = SUPERVISED_ROOT / f"confusion_matrix_{run_name}.png"
    subject_table_path = SUPERVISED_ROOT / f"per_subject_accuracy_{run_name}.csv"

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "best_epoch": best_epoch,
            "label_to_index": LABEL_TO_INDEX,
            "manifest_path": str(MANIFEST_PATH),
            "run_name": run_name,
        },
        checkpoint_path,
    )

    metrics_payload = {
        "run_name": run_name,
        "best_epoch": best_epoch,
        "config": {
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "dropout": DROPOUT,
            "epochs_requested": max_epochs,
        },
        "train": {
            "loss": train_metrics["loss"],
            "accuracy": train_metrics["accuracy"],
            "macro_f1": train_metrics["macro_f1"],
        },
        "val": {
            "loss": val_metrics["loss"],
            "accuracy": val_metrics["accuracy"],
            "macro_f1": val_metrics["macro_f1"],
        },
        "test": {
            "loss": test_metrics["loss"],
            "accuracy": test_metrics["accuracy"],
            "macro_f1": test_metrics["macro_f1"],
            "per_class_f1": test_metrics["per_class_f1"],
            "per_subject_accuracy": test_metrics["per_subject_accuracy"],
        },
        "history": history,
    }
    metrics_path.write_text(json.dumps(metrics_payload, indent=2, sort_keys=True), encoding="utf-8")

    with subject_table_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=["subject_id", "accuracy"])
        writer.writeheader()
        for subject_id, accuracy in test_metrics["per_subject_accuracy"].items():
            writer.writerow({"subject_id": subject_id, "accuracy": accuracy})

    plot_confusion_matrix(test_metrics["confusion_matrix"], f"Supervised Baseline ({run_name})", confusion_path)

    return {
        "checkpoint_path": str(checkpoint_path),
        "metrics_path": str(metrics_path),
        "confusion_path": str(confusion_path),
        "subject_table_path": str(subject_table_path),
        "metrics": metrics_payload,
    }


In [ ]:
results = train_supervised_baseline()
print("\nArtifacts:")
print(json.dumps({k: v for k, v in results.items() if k.endswith('_path')}, indent=2))
print("\nTest metrics:")
print(json.dumps(results["metrics"]["test"], indent=2))

top_subjects = list(results["metrics"]["test"].get("per_subject_accuracy", {}).items())[:5]
if top_subjects:
    print("\nPer-subject accuracy preview:")
    for subject_id, accuracy in top_subjects:
        print(subject_id, round(accuracy, 4))